# No-mapping baseline and direct mapped-policy rollout

First evaluate the trajectory buffer with the identity mapping $z'=z$.

For each sample, the baseline uses the same discrepancy as mapping training:

$$d = \operatorname{mean}_{t,xy}\left[(y-Az)^2s + 25(1-s)\right].$$

Because $F$ is exactly the identity and this test uses $p=q$, the importance ratio is exactly one, $D_{KL}(q'\|q)=0$, and the effective-sample-size fraction is one. The full buffer is shuffled and evaluated in batches to avoid creating the decoded trajectories for every sample at once.

The second part loads the learned mapping and policy, samples desired targets $z'\sim p$, forward-maps them to policy inputs $z=F(z')$, and measures the same discrepancy directly from fresh rollouts without importance weighting.

In [ ]:
from pathlib import Path
import json
import os
import time

import jax
import jax.numpy as jnp
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jax_rq_nsf").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SEED = 8848
BATCH_SIZE = int(os.environ.get("TBQDRL_BASELINE_BATCH_SIZE", "4096"))
REPORT_PERIOD = int(os.environ.get("TBQDRL_BASELINE_REPORT_PERIOD", "32"))
DEATH_ERROR = 10.0
TRAJECTORY_BUFFER_PATH = Path(
    os.environ.get(
        "TBQDRL_TRAJECTORY_BUFFER",
        PROJECT_ROOT / "output/mapping_test/trajectory_buffer.npz",
    )
)
PROJECTION_MATRIX_PATH = (
    PROJECT_ROOT / "output/mapping_test/trajectory_projection_matrix.npy"
)
OUTPUT_PATH = Path(
    os.environ.get(
        "TBQDRL_BASELINE_OUTPUT",
        PROJECT_ROOT / "output/mapping_test/no_mapping_baseline.json",
    )
)

assert BATCH_SIZE > 0
assert REPORT_PERIOD > 0
print("JAX devices:", jax.devices())
print("Batch size:", BATCH_SIZE)

In [ ]:
assert TRAJECTORY_BUFFER_PATH.is_file(), TRAJECTORY_BUFFER_PATH
assert PROJECTION_MATRIX_PATH.is_file(), PROJECTION_MATRIX_PATH

with np.load(TRAJECTORY_BUFFER_PATH, allow_pickle=False) as archive:
    buffer_z = archive["z"]
    buffer_xy = archive["xy"]
    buffer_scale = archive["scale"]

projection_matrix = np.load(PROJECTION_MATRIX_PATH, allow_pickle=False)
projection_matrix = jnp.asarray(projection_matrix, dtype=jnp.float32)

num_samples = buffer_z.shape[0]
assert buffer_z.shape == (num_samples, 50)
assert buffer_xy.shape == (num_samples, 96, 2)
assert buffer_scale.shape == (num_samples, 96, 1)
assert projection_matrix.shape == (96, 25)
assert num_samples >= BATCH_SIZE

print("Trajectory samples:", f"{num_samples:,}")
print("Projection matrix:", projection_matrix.shape, projection_matrix.dtype)

In [ ]:
@jax.jit
def evaluate_batch(z, xy, scale):
    per_axis_z = z.reshape(z.shape[:-1] + (2, 25))
    target_xy = jnp.einsum(
        "td,...cd->...tc",
        projection_matrix,
        per_axis_z,
    )
    tracking_contribution = jnp.mean(
        jnp.square(xy - target_xy) * scale,
        axis=(-1, -2),
    )
    termination_contribution = jnp.mean(
        (1.0 - scale) * DEATH_ERROR,
        axis=(-1, -2),
    )
    discrepancy = tracking_contribution + termination_contribution
    return discrepancy, tracking_contribution, termination_contribution

In [ ]:
rng = np.random.default_rng(SEED)
shuffled_indices = rng.permutation(num_samples)
num_batches = int(np.ceil(num_samples / BATCH_SIZE))

per_sample_discrepancy = np.empty((num_samples,), dtype=np.float32)
per_sample_tracking = np.empty((num_samples,), dtype=np.float32)
per_sample_termination = np.empty((num_samples,), dtype=np.float32)

start_time = time.monotonic()
offset = 0
for batch_index in range(num_batches):
    indices = shuffled_indices[offset : offset + BATCH_SIZE]
    discrepancy, tracking, termination = jax.device_get(
        evaluate_batch(
            buffer_z[indices],
            buffer_xy[indices],
            buffer_scale[indices],
        )
    )
    stop = offset + indices.size
    per_sample_discrepancy[offset:stop] = discrepancy
    per_sample_tracking[offset:stop] = tracking
    per_sample_termination[offset:stop] = termination
    offset = stop

    completed = batch_index + 1
    if completed == 1 or completed % REPORT_PERIOD == 0 or completed == num_batches:
        elapsed = time.monotonic() - start_time
        running_mse = float(np.mean(per_sample_discrepancy[:offset]))
        print(
            f"batch {completed:4d}/{num_batches} | "
            f"running MSE {running_mse:.6f} | "
            f"{elapsed:.1f}s"
        )

assert offset == num_samples

In [ ]:
summary = {
    "num_samples": int(num_samples),
    "batch_size": int(BATCH_SIZE),
    "mse_loss": float(np.mean(per_sample_discrepancy)),
    "mse_std": float(np.std(per_sample_discrepancy)),
    "tracking_mse_contribution": float(np.mean(per_sample_tracking)),
    "termination_penalty_contribution": float(
        np.mean(per_sample_termination)
    ),
    "masked_step_fraction": float(
        np.mean(per_sample_termination) / DEATH_ERROR
    ),
    "mse_percentiles": {
        str(percentile): float(
            np.percentile(per_sample_discrepancy, percentile)
        )
        for percentile in (0, 25, 50, 75, 90, 95, 99, 100)
    },
    "kl_divergence": 0.0,
    "importance_mean": 1.0,
    "importance_max": 1.0,
    "ess_fraction": 1.0,
    "trajectory_buffer": str(TRAJECTORY_BUFFER_PATH),
    "projection_matrix": str(PROJECTION_MATRIX_PATH),
}

print(json.dumps(summary, indent=2))
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(summary, indent=2) + "\n")
print("Saved baseline:", OUTPUT_PATH)

## Direct rollout with the learned mapping

Sample desired static trajectory encodings from $p$, apply the learned forward map to obtain policy inputs, and supply those static encodings to the policy wrapper. The policy still receives its full 55-dimensional online conditioning vector after the wrapper adds phase, task time, and deviation features.

In [ ]:
import sys

import flax.linen as nn
from brax import envs
from flax import serialization

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jax_rq_nsf import FlowConfig, init_flow, init_invertible_nsf
from networks import PPO_Policy
from task_wrappers.nf_trajectory_following import AntFiniteMaternWrapper

DIRECT_ROLLOUTS = int(os.environ.get("TBQDRL_DIRECT_ROLLOUTS", "8192"))
FLOW_CHECKPOINT_PATH = (
    PROJECT_ROOT / "output/vi_filtered/trajectory_flow_variables.msgpack"
)
POLICY_CHECKPOINT_PATH = (
    PROJECT_ROOT / "output/vi_filtered_ppo/saved_data/policy.msgpack"
)
MAPPING_CHECKPOINT_PATH = Path(
    os.environ.get(
        "TBQDRL_MAPPING_CHECKPOINT",
        PROJECT_ROOT / "output/mapping_test/mapping_variables.msgpack",
    )
)

assert DIRECT_ROLLOUTS > 0
assert FLOW_CHECKPOINT_PATH.is_file(), FLOW_CHECKPOINT_PATH
assert POLICY_CHECKPOINT_PATH.is_file(), POLICY_CHECKPOINT_PATH
assert MAPPING_CHECKPOINT_PATH.is_file(), MAPPING_CHECKPOINT_PATH

model_config = FlowConfig(
    dimension=50,
    num_layers=8,
    hidden_features=(128, 128),
    num_bins=16,
    tail_bound=6.0,
)
rollout_key = jax.random.PRNGKey(SEED + 10)
rollout_key, flow_key, mapping_key, policy_key = jax.random.split(
    rollout_key,
    4,
)

flow, flow_template = init_flow(flow_key, model_config)
flow_variables = serialization.from_bytes(
    flow_template,
    FLOW_CHECKPOINT_PATH.read_bytes(),
)
mapping_config = FlowConfig(
    dimension=50,
    num_layers=4,
    hidden_features=(128, 128),
    num_bins=16,
    tail_bound=4.0,
)


mapping, mapping_template = init_invertible_nsf(mapping_key, mapping_config)
mapping_variables = serialization.from_bytes(
    mapping_template,
    MAPPING_CHECKPOINT_PATH.read_bytes(),
)

base_env = envs.create(
    env_name="ant",
    episode_length=4096,
    backend="mjx",
    reset_noise_scale=0.0,
)
env = AntFiniteMaternWrapper(
    base_env,
    flow,
    way_points=12,
    steps_per_way_point=8,
    inner_radius=0.5,
    max_radius=2.0,
    tolerance_radius=0.1,
)
policy = PPO_Policy(
    hidden_layer_sizes=(256, 256),
    action_dim=env.action_size,
    kernel_init=jax.nn.initializers.orthogonal(jnp.sqrt(2.0)),
    kernel_init_final=jax.nn.initializers.orthogonal(0.01),
    activation=nn.silu,
    final_activation=jnp.tanh,
)
policy_template = policy.init(
    policy_key,
    obs=jnp.zeros((env.observation_size,)),
    z=jnp.zeros((env.z_size,)),
)
policy_variables = serialization.from_bytes(
    policy_template,
    POLICY_CHECKPOINT_PATH.read_bytes(),
)
action_std = nn.sigmoid(policy_variables["params"]["std_logits"])

print("Direct mapped rollouts:", DIRECT_ROLLOUTS)
print("Flow:", FLOW_CHECKPOINT_PATH)
print("Policy:", POLICY_CHECKPOINT_PATH)
print("Mapping:", MAPPING_CHECKPOINT_PATH)

In [ ]:
def install_static_task(state, static_z):
    sampled_sequence = static_z.reshape(2, 25)
    position_offset = -env._get_position_from_envstate(state.env_state)
    task_state = env._organize_z_state(sampled_sequence, position_offset)
    return state.replace(
        z_state=task_state,
        initial_z_state=task_state,
    )


@jax.jit
def collect_mapped_rollouts(key):
    target_key, reset_key, action_key = jax.random.split(key, 3)
    target_z = flow.apply(
        flow_variables,
        target_key,
        DIRECT_ROLLOUTS,
        method=flow.sample,
    )
    policy_z, forward_log_det = mapping.apply(
        mapping_variables,
        target_z,
        method=mapping.forward,
    )

    reset_keys = jax.random.split(reset_key, DIRECT_ROLLOUTS)
    states = jax.vmap(env.reset, in_axes=(0, None))(
        reset_keys,
        flow_variables,
    )
    states = jax.vmap(install_static_task)(states, policy_z)
    initial_xy = states.env_state.pipeline_state.x.pos[:, 0, :2]

    def rollout_step(carry, _):
        current_states, alive, step_key = carry
        observations, conditioning = env.get_obs(current_states)
        action_mean, _ = policy.apply(
            policy_variables,
            observations,
            conditioning,
        )
        step_key, noise_key = jax.random.split(step_key)
        actions = action_mean + action_std * jax.random.normal(
            noise_key,
            action_mean.shape,
        )
        next_states, transition_info = jax.vmap(env.step)(
            current_states,
            jnp.clip(actions, -1.0, 1.0),
        )
        terminated = transition_info.done[..., 0] > 0.5
        alive = alive & ~terminated
        xy = (
            next_states.env_state.pipeline_state.x.pos[:, 0, :2]
            - initial_xy
        )
        scale = alive.astype(xy.dtype)[..., None]
        return (next_states, alive, step_key), (xy, scale)

    def rollout_period(carry, _):
        carry, step_data = jax.lax.scan(
            rollout_step,
            carry,
            xs=None,
            length=8,
        )
        current_states, alive, step_key = carry
        current_states = jax.vmap(env.shift)(current_states)
        return (current_states, alive, step_key), step_data

    initial_alive = jnp.ones((DIRECT_ROLLOUTS,), dtype=jnp.bool_)
    _, (xy_blocks, scale_blocks) = jax.lax.scan(
        rollout_period,
        (states, initial_alive, action_key),
        xs=None,
        length=12,
    )
    xy = jnp.swapaxes(
        xy_blocks.reshape(96, DIRECT_ROLLOUTS, 2),
        0,
        1,
    )
    scale = jnp.swapaxes(
        scale_blocks.reshape(96, DIRECT_ROLLOUTS, 1),
        0,
        1,
    )
    return target_z, policy_z, forward_log_det, xy, scale

In [ ]:
rollout_key = jax.random.PRNGKey(22)

In [ ]:
rollout_key, direct_rollout_key = jax.random.split(rollout_key)
(
    target_z,
    policy_z,
    forward_log_det,
    mapped_xy,
    mapped_scale,
) = collect_mapped_rollouts(direct_rollout_key)

mapped_discrepancy, mapped_tracking, mapped_termination = jax.device_get(
    evaluate_batch(target_z, mapped_xy, mapped_scale)
)
p_log_prob = flow.apply(
    flow_variables,
    target_z,
    method=flow.log_prob,
)
projected_log_prob = p_log_prob - forward_log_det
q_log_prob = flow.apply(
    flow_variables,
    policy_z,
    method=flow.log_prob,
)
direct_kl = float(jnp.mean(projected_log_prob - q_log_prob))

mapped_mse = float(np.mean(mapped_discrepancy))
mapping_summary = {
    "num_rollouts": int(DIRECT_ROLLOUTS),
    "mse_loss": mapped_mse,
    "mse_std": float(np.std(mapped_discrepancy)),
    "tracking_mse_contribution": float(np.mean(mapped_tracking)),
    "termination_penalty_contribution": float(
        np.mean(mapped_termination)
    ),
    "masked_step_fraction": float(
        np.mean(mapped_termination) / DEATH_ERROR
    ),
    "mse_percentiles": {
        str(percentile): float(
            np.percentile(mapped_discrepancy, percentile)
        )
        for percentile in (0, 25, 50, 75, 90, 95, 99, 100)
    },
    "direct_kl_divergence": direct_kl,
    "mse_reduction_from_buffer_baseline": float(
        summary["mse_loss"] - mapped_mse
    ),
    "relative_mse_reduction": float(
        1.0 - mapped_mse / summary["mse_loss"]
    ),
    "mapping_checkpoint": str(MAPPING_CHECKPOINT_PATH),
    "flow_checkpoint": str(FLOW_CHECKPOINT_PATH),
    "policy_checkpoint": str(POLICY_CHECKPOINT_PATH),
}
summary["direct_mapped_rollout"] = mapping_summary

print(json.dumps(mapping_summary, indent=2))
OUTPUT_PATH.write_text(json.dumps(summary, indent=2) + "\n")
print("Updated comparison:", OUTPUT_PATH)